In [2]:
import os
import pandas as pd

In [6]:
# Define project directory
root = os.path.abspath(os.path.join(os.getcwd(), '..'))
print(f"Project root: {root}")

Project root: C:\Users\sjdahlke\Documents\2025 Plant Costs


In [7]:
# Load data
path = os.path.join(root, 'data', 'ferc1_plants_utilitynames.csv')
ferc = pd.read_csv(path)
df1_cols = pd.read_csv(path).columns.tolist()
print("Columns in cost_data_v1.csv:", df1_cols)

path = os.path.join(root, 'data', 'eia_gen.csv')
eia = pd.read_csv(path, low_memory=False)

path = os.path.join(root, 'data', 'core_ferc1__yearly_steam_plants_sched402.csv')
df2_cols = pd.read_csv(path).columns.tolist()
print("Columns in core_ferc1__yearly_steam_plants_sched402.csv:", df2_cols)

Columns in cost_data_v1.csv: ['rowid', 'record_id', 'utility_id_ferc1', 'utility_id_ferc1_label', 'report_year', 'plant_name_ferc1', 'plant_id_eia', 'plant_name_eia', 'generator_id', 'units', 'data_questionable', 'retired', 'plant_type', 'construction_type', 'construction_year', 'installation_year', 'capacity_mw', 'peak_demand_mw', 'plant_hours_connected_while_generating', 'plant_capability_mw', 'water_limited_capacity_mw', 'not_water_limited_capacity_mw', 'avg_num_employees', 'net_generation_mwh', 'capex_land', 'capex_structures', 'capex_equipment', 'capex_total', 'capex_per_mw', 'opex_operations', 'opex_fuel', 'opex_coolants', 'opex_steam', 'opex_steam_other', 'opex_transfer', 'opex_electric', 'opex_misc_power', 'opex_rents', 'opex_allowances', 'opex_engineering', 'opex_structures', 'opex_boiler', 'opex_plants', 'opex_misc_steam', 'opex_production_total', 'opex_per_mwh', 'asset_retirement_cost', 'cleaningNotes']
Columns in core_ferc1__yearly_steam_plants_sched402.csv: ['rowid', 'reco

## Match historical cost data into 2023 dataset

In [8]:

# --- Configuration ---
D1_PATH = os.path.join(root, 'data', 'ferc1_plants_utilitynames.csv')
D2_PATH = os.path.join(root, 'data', 'core_ferc1__yearly_steam_plants_sched402.csv')
CURRENT_YEAR_OF_TRUTH = 2023 # Year for which D1 is the primary source

# Identifiers
PLANT_NAME_FERC1_COL = 'plant_name_ferc1'
PLANT_ID_EIA_COL = 'plant_id_eia'
GENERATOR_ID_COL = 'generator_id'
UTILITY_ID_FERC1_COL = 'utility_id_ferc1'
REPORT_YEAR_COL = 'report_year'

# Additional D1-specific columns to carry over for the CURRENT_YEAR_OF_TRUTH
# (These will be NaN for historical years from D2 unless also present and selected from D2)
D1_SPECIFIC_ATTR_COLS = [
    'units',
    'data_questionable',
    'retired'
]

# Year-specific attributes (common to both, values taken from source year)
YEARLY_ATTRIBUTE_COLS = [
    'plant_type',
    'construction_type',
    'construction_year',
    'installation_year',
    'capacity_mw',
    'peak_demand_mw',
    'plant_hours_connected_while_generating',
    'plant_capability_mw',
    'water_limited_capacity_mw',
    'not_water_limited_capacity_mw',
    'avg_num_employees',
    'net_generation_mwh'
]

# Cost columns (common to both, values taken from source year)
# These names are identical in both files.
GENERIC_COST_COLS = [
    'capex_land', 'capex_structures', 'capex_equipment', 'capex_total', 'capex_per_mw',
    'opex_operations', 'opex_fuel', 'opex_coolants', 'opex_steam', 'opex_steam_other',
    'opex_transfer', 'opex_electric', 'opex_misc_power', 'opex_rents', 'opex_allowances',
    'opex_engineering', 'opex_structures', # This is the opex_structures
    'opex_boiler', 'opex_plants', 'opex_misc_steam', 'opex_production_total', 'opex_per_mwh',
    'asset_retirement_cost'
]


# --- Load DataFrames ---
print(f"Loading Dataset 1 from: {D1_PATH}")
df1 = pd.read_csv(D1_PATH)
print(f"Loading Dataset 2 from: {D2_PATH}")
df2 = pd.read_csv(D2_PATH)

print(f"\nInitial D1 shape: {df1.shape}, Initial D2 shape: {df2.shape}")



Loading Dataset 1 from: C:\Users\sjdahlke\Documents\2025 Plant Costs\data\ferc1_plants_utilitynames.csv
Loading Dataset 2 from: C:\Users\sjdahlke\Documents\2025 Plant Costs\data\core_ferc1__yearly_steam_plants_sched402.csv

Initial D1 shape: (959, 48), Initial D2 shape: (32975, 40)


In [9]:

# --- Step 1: Prepare D1 (Current Year Data) ---
print(f"\n--- Preparing Dataset 1 for year {CURRENT_YEAR_OF_TRUTH} ---")
# Filter D1 for the current year of truth
df1_current_year = df1[df1[REPORT_YEAR_COL] == CURRENT_YEAR_OF_TRUTH].copy()
print(f"D1 rows for {CURRENT_YEAR_OF_TRUTH}: {len(df1_current_year)}")

# Define columns to select from D1 for the current year
d1_cols_to_select = \
    [PLANT_NAME_FERC1_COL, PLANT_ID_EIA_COL, GENERATOR_ID_COL, UTILITY_ID_FERC1_COL, REPORT_YEAR_COL] + \
    [col for col in D1_SPECIFIC_ATTR_COLS if col in df1_current_year.columns] + \
    [col for col in YEARLY_ATTRIBUTE_COLS if col in df1_current_year.columns] + \
    [col for col in GENERIC_COST_COLS if col in df1_current_year.columns]
df1_current_year_data = df1_current_year[list(dict.fromkeys(d1_cols_to_select))] # Select unique columns in order

print(f"D1 Prepared Data (current year {CURRENT_YEAR_OF_TRUTH}) - Shape: {df1_current_year_data.shape}")
print(df1_current_year_data.head())


--- Preparing Dataset 1 for year 2023 ---
D1 rows for 2023: 959
D1 Prepared Data (current year 2023) - Shape: (959, 43)
      plant_name_ferc1  plant_id_eia generator_id  utility_id_ferc1  \
0  rockport unit 1 aeg        6166.0            1               342   
1  rockport unit 2 aeg        6166.0            2               342   
2                barry           3.0            5               294   
3             barry cc           3.0         A1CT               294   
4              calhoun       55409.0         CAL1               294   

   report_year  units  data_questionable  retired          plant_type  \
0         2023    NaN                NaN      NaN               steam   
1         2023    NaN                NaN      NaN               steam   
2         2023    NaN                NaN      NaN               steam   
3         2023    NaN                NaN      NaN      combined_cycle   
4         2023    NaN                NaN      NaN  combustion_turbine   

  constructio

In [10]:

# --- Step 2: Prepare D2 (Historical Data) ---
print(f"\n--- Preparing Dataset 2 for years before {CURRENT_YEAR_OF_TRUTH} ---")
df2_historical = df2[df2[REPORT_YEAR_COL] < CURRENT_YEAR_OF_TRUTH].copy()
print(f"D2 historical rows (year < {CURRENT_YEAR_OF_TRUTH}): {len(df2_historical)}")

# Define columns to select from D2 for historical years
d2_cols_to_select = \
    [PLANT_NAME_FERC1_COL, UTILITY_ID_FERC1_COL, REPORT_YEAR_COL] + \
    [col for col in YEARLY_ATTRIBUTE_COLS if col in df2_historical.columns] + \
    [col for col in GENERIC_COST_COLS if col in df2_historical.columns]
df2_historical_data = df2_historical[list(dict.fromkeys(d2_cols_to_select))]

print(f"D2 Prepared Data (historical) - Shape: {df2_historical_data.shape}")
print(df2_historical_data.head())





--- Preparing Dataset 2 for years before 2023 ---
D2 historical rows (year < 2023): 31879
D2 Prepared Data (historical) - Shape: (31879, 38)
     plant_name_ferc1  utility_id_ferc1  report_year plant_type  \
0            (1)sta98               214         1994      steam   
1  (2) scriba sta. 99               214         1994    nuclear   
2    (n) contra costa               183         1995      steam   
3       (n) pittsburg               183         1995      steam   
4  (n)(s)contra costa               183         1994      steam   

  construction_type  construction_year  installation_year  capacity_mw  \
0      conventional             1979.0             1979.0        216.4   
1               NaN             1988.0             1988.0        169.9   
2       semioutdoor             1951.0             1964.0        718.0   
3       semioutdoor             1954.0             1972.0       2028.9   
4       semioutdoor             1951.0             1964.0        718.0   

   peak_de

In [11]:

# --- Step 3: Add Key Identifiers (PLANT_ID_EIA, GENERATOR_ID) to D2's Historical Data ---
print("\n--- Adding D1 Key Identifiers to D2 Historical Data ---")
# Create a mapping table from D1 (current year) for the key identifiers
# We use df1_current_year_data to ensure we map IDs from the "source of truth" year
identifiers_map_df = df1_current_year_data[[
    PLANT_NAME_FERC1_COL,
    PLANT_ID_EIA_COL,
    GENERATOR_ID_COL
]].drop_duplicates(subset=[PLANT_NAME_FERC1_COL])
print(f"Identifier map created with {len(identifiers_map_df)} unique plants from D1 ({CURRENT_YEAR_OF_TRUTH}).")

# Merge these identifiers into the historical data from D2
df2_historical_with_ids = pd.merge(
    df2_historical_data,
    identifiers_map_df,
    on=PLANT_NAME_FERC1_COL,
    how='left' # Keep all historical records from D2, add IDs if plant exists in D1 map
)
print(f"D2 Historical Data with IDs merged - Shape: {df2_historical_with_ids.shape}")
print(df2_historical_with_ids.head())
# Check how many historical records got an EIA ID
print(f"Historical records with matched {PLANT_ID_EIA_COL}: {df2_historical_with_ids[PLANT_ID_EIA_COL].notna().sum()}")


--- Adding D1 Key Identifiers to D2 Historical Data ---
Identifier map created with 925 unique plants from D1 (2023).
D2 Historical Data with IDs merged - Shape: (31879, 40)
     plant_name_ferc1  utility_id_ferc1  report_year plant_type  \
0            (1)sta98               214         1994      steam   
1  (2) scriba sta. 99               214         1994    nuclear   
2    (n) contra costa               183         1995      steam   
3       (n) pittsburg               183         1995      steam   
4  (n)(s)contra costa               183         1994      steam   

  construction_type  construction_year  installation_year  capacity_mw  \
0      conventional             1979.0             1979.0        216.4   
1               NaN             1988.0             1988.0        169.9   
2       semioutdoor             1951.0             1964.0        718.0   
3       semioutdoor             1954.0             1972.0       2028.9   
4       semioutdoor             1951.0             1

In [12]:
# --- Step 4: Combine D1 (Current Year) and Processed D2 (Historical with IDs) ---
print("\n--- Combining D1 (Current Year) and D2 (Historical with IDs) ---")
# Concatenate the two dataframes. Pandas will align columns by name.
# Columns present in one but not the other will be filled with NaN.
final_df = pd.concat([df1_current_year_data, df2_historical_with_ids], ignore_index=True, sort=False)
print(f"Combined DataFrame - Shape: {final_df.shape}")



--- Combining D1 (Current Year) and D2 (Historical with IDs) ---
Combined DataFrame - Shape: (32838, 43)


In [15]:
# --- Step 5: Final Touches ---
print("\n--- Applying Final Touches ---")

# Ensure correct data types
if REPORT_YEAR_COL in final_df.columns:
    final_df[REPORT_YEAR_COL] = final_df[REPORT_YEAR_COL].astype(int)
if PLANT_ID_EIA_COL in final_df.columns:
    final_df[PLANT_ID_EIA_COL] = final_df[PLANT_ID_EIA_COL].astype(pd.Int64Dtype()) # Supports NA for integers
# Convert other numeric columns if necessary, e.g. capacity_mw, cost columns to float
for col in YEARLY_ATTRIBUTE_COLS + GENERIC_COST_COLS:
    if col in final_df.columns:
        final_df[col] = pd.to_numeric(final_df[col], errors='coerce')


# Sort the data
final_df = final_df.sort_values(
    by=[PLANT_NAME_FERC1_COL, REPORT_YEAR_COL],
    ascending=[True, False] # Sort by plant name, then by year descending
).reset_index(drop=True)

print("\n--- Final Combined and Cleaned DataFrame ---")
print(final_df.head()) # Using to_string() to see more of the output
print(f"\nShape of final DataFrame: {final_df.shape}")
print("\nData types of final DataFrame:")
print(final_df.dtypes)

# --- Optional: Save to CSV ---
OUTPUT_PATH = os.path.join(root, 'data', 'plant_costs_with_history.csv')
final_df.to_csv(OUTPUT_PATH, index=False)
print(f"\nFinal DataFrame saved to {OUTPUT_PATH}")



--- Applying Final Touches ---

--- Final Combined and Cleaned DataFrame ---
     plant_name_ferc1  plant_id_eia generator_id  utility_id_ferc1  \
0            (1)sta98          <NA>          NaN               214   
1  (2) scriba sta. 99          <NA>          NaN               214   
2    (n) contra costa          <NA>          NaN               183   
3       (n) pittsburg          <NA>          NaN               183   
4  (n)(s)contra costa          <NA>          NaN               183   

   report_year  units  data_questionable  retired  plant_type  \
0         1994    NaN                NaN      NaN         NaN   
1         1994    NaN                NaN      NaN         NaN   
2         1995    NaN                NaN      NaN         NaN   
3         1995    NaN                NaN      NaN         NaN   
4         1994    NaN                NaN      NaN         NaN   

   construction_type  ...  opex_rents  opex_allowances  opex_engineering  \
0                NaN  ...    34820

In [16]:
# match datasets on plant_id_eia, generator_id
df = pd.merge(final_df, eia, on=['plant_id_eia', 'generator_id'], how='left')

In [17]:
#clean needed things

#incorrect sign on longitude for one plant:
mask = df['plant_name_ferc1'] == 'ferndale'
df.loc[mask, 'longitude'] = -df.loc[mask, 'longitude']

# Add missing lat/longs as needed
mask = df['plant_name_ferc1'] == 'maryland heights lf'
df.loc[mask, ['technology_description','latitude','longitude']] = ['Municipal Solid Waste', 38.6828611,-90.4878889]

mask = df['plant_name_ferc1'] == 'coxsackie'
df.loc[mask, ['latitude']] = 42.350889

mask = df['plant_name_ferc1'] == 'whitehorn'
df.loc[mask, ['latitude']] = 48.8857982

# Add missing capacity from FERC data
mask = df['plant_name_ferc1'] == 'clemson chp'
df.loc[mask, ['capacity_mw_x']] = 15

In [18]:
df.to_csv(os.path.join(root, 'data', 'cost_data_v1.csv'), index=False)